# 에이전트 Agent
* 다음 행동을 스스로 고르는 실행 구조

## 1. Chain vs Agent

* Chain
    * 사람이 코드로 고정
    * 모델 호출 횟수:  보통 1번
    * 결과물: 파서를 붙이면 문자열 하나
    * 예측 가능성 높음 - 항상 같은 경로
    * 요약, 번역, 분류처럼 정해진 변환

* Agent
    * 모델이 실행중에 결정
    * 호출횟수: 필요한 만큼(Tool 쓰면 2번 이상)
    * 결과물: messages 목록 (중간 과정 포함)
    * 예측 가능성 낮음 - 같은 질문도 다른 경로 탈 수 있음
    * 상황에 따라 다르게 처리해야 하는 일
    * 디버깅 어려움 - stream() 배우기

## 2. Tool
* 모델만으로 해결하기 어려운 문제를 처리하기 위해 바깥 정보나 기능을 연결하는 장치

### 2-1. Tool vs Structured Output
* Tool
    * 모델이 모르는 것을 알게 하기
    * 답을 만드는 중간에 실행됨
    * tools=[...]
* Structured Output
    * 답을 프로그램이 쓸 수 있는 모양으로 받기
    * 답을 만드는 마지막에 실행됨
    * response_format=...

--> 서로 따로 써도 되고 같이 써도 됨

### 2-2. Tool을 잘 정의하는 4가지 원칙
1. 이름 - 기능이 바로 드러나게
    * 모델은 이름을 보고도 반쯤 판단함
    * lookup (X) / lookup_order_status (O)

2. 입력 - 필요한 값을 필드로 나누기
    * 칸이 나눠져있으면 모델이 무엇을 채워야 할 지 앎
    * query: str (X) / order_id, date 등

3. docstring - 언제 써야하는지 짧고 분명하게
    * "데이터를 조회합니다" (X) / "주문번호로 배송 상태를 조회합니다" (O)

4. 반환값과 실패 메시지 - 애매하지 않게
    * 다음 단계(모델)가 판단하기 쉽게 돌려주기
    * 못 찾았으면: None (X) / "해당 주문번호를 찾을 수 없습니다" (O)

### 2-3. @tool 문법


In [ ]:
from langchain.tools import tool

@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송 상태를 확인해야 할 때 사용합니다."""
    return "배송 중입니다. 오늘 오후 도착 예정입니다."

### 2-4.입력이 복잡해지면 args_schema

In [ ]:
from pydantic import BaseModel, Field
from langchain.tools import tool

class CustomerRequestInput(BaseModel):
    request: str = Field(description="분류할 고객 문의")

@tool(args_schema=CustomerRequestInput)
def classify_customer_request(request:str) -> str:
    """고객 문의 내용을 읽고 배송, 교환, 환불 중 하나로 분류해야 할 떄 사용합니다."""

## 3.create_agent - 4줄로 만드는 에이전트

### 3-1. 기본형

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    # 호출할 채팅 모델
    model=model,
    
    # 에이전트가 호출할 수 있는 tool 목록
    tools=[lookup_order_status],
    
    # 모델에게 먼저 주는 역할과 규칙
    system_prompt="주문 배송 상태 질문은 lookup_order_status 도구로 확인한 뒤 답하세요."
    
    # 실행 중간에 끼워 넣는 처리
    middleware=

    # 최종 답변을 특정 구조로 받기 위한 스키마
    response_format=

    # 에이전트 상태에 추가로 저장할 값의 구조
    state_schema=

    # 실행될 때 함께 넘기는 런타임 정보의 구조
    context_schema=

    # 에이전트 상태를 저장하고 다시 불러오는 저장장치
    checkpointer=
)

### 3-2. 실행결과: 딕셔너리 (문자열 아님)
* messages가 들어있는 딕셔너리를 돌려줌 - 디버깅을 할 수 있도록 과정을 통째로 돌려줌

In [ ]:
# 전체 메시지를 보면 Tool 호출과 Tool 실행 결과가 어떤 순서로 쌓였는지 확인할 수 있음
for index, message in enumerate(response["messaes"], start=1):
    print(f"\n[{index}] {type(message).__name__}")
    print("content:", repr(message.content))

    if getattr(message, "tool_calls", None):
        print("tool_calls:", message.tool_calls)

    if getattr(message, "tool_call_id", None):
        print("tool_call_id:", message.tool_call_id)

### 3-3. 메시지 4개 쌓이는 순서
1. HumanMessage: 사용자 질문
    * 질문이 제대로 들어갔는지 ?
2. AIMessage: 모델이 Tool 호출 결정함
    * tool_calls - 어떤 Tool을, 어떤 인자로 부르려 했는지 ?
    * 아예 없다면 모델이 Tool 쓸 생각을 안함 - docstring이나 system_prompt문제
    * args가 이상하다면 - 입력 스키마가 흐림
3. ToolMessage: Tool 실행 결과
    * Tool이 무엇을 돌려줬는지 ?
    * 내용이 이상하다면 - 내 함수 코드 버그.
4. AIMessage: Tool 결과를 반영한 최종 답변
    * 최종 답변 ```reponse["message"][-1].content```
    * 3이 멀쩡한데 4가 엉뚱하다면 - 모델이 Tool 결과를 제대로 못 읽음. 반환 형식을 더 단순하게 변경필요

## 4 Structured Output - 문장 대신 칸으로 받기
* 모델의 답을 사람이 읽는 문장이 아닌 프로그램이 바로 쓸 수 있는 데이터로 받음

## 5. 여러 Tool과 stream()

### 5-1. Tool 두 개 준비

In [ ]:
# tool 두 개를 등록해 에이전트가 상황에 맞게 직접 선택하게 만듦
from datetime import datetime # 현재 시각
from zoneinfo import ZoneInfo # 서울 시간대 지정
from langchain.agents import create_aget # Tool을 쓸 수 있는 에이전트 생성
from langchain.tools import tool 

@tool
def get_korean_time() -> str:
    """현재 서울 시간을 반환합니다."""
    now = datetime.now(ZoneInfo("Asia/Seoul"))
    return now.strftime("%Y-%m-%d %H:%M:%S")

@tool
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더합니다. 계산이 필요할 때 사용하세요"""
    return a + b

### 5-2. 한 질문에 Tool 두 개

In [ ]:
# 두 Tool을 에이전트에 연결
basic_agent = create_agent(
    model=model,
    tools=[get_korean_time, add_numbers],
    system_prompt="당신은 쇼핑몰 운영 보조 에이전트입니다. 시간과 계산이 필요하면 반드시 도구를 사용하세요.",
)

# 한 질문 안에서 시간 Tool과 계산 Tool이 함께 쓰이는지 확인
result = basic_agent.invoke(
    {"messgaes"; [{"role": "user", "content": "서울 기준 현재 시간과 쿠폰 3000원 + 적립금 2000원 합계를 알려줘."}]},
    lf_config,
)

### 5-3. stream() - 실행중 업데이트를 순서대로 받을 수 있음
* 한 단계가 끝날 때마다 바로 나옴

In [ ]:
# stream_mode="updates"는 실행 중 업데이트를 단계별로 보여줌
for chunk in basic_agent.stream(
    {"messages": [{"role": "user", "content": "지금 시간만 알려줘."}]},
    stream_mode="updates",
    config=lf_config
)

print(chunk)

## 6. 메모리 - checkpointer와 thread_id
* 대화가 이어진다는 것은 단순히 같은 에이전트를 다시 호출한다는 뜻이 아닌, 이전 대화 상태를 다음 요청의 Context에 다시 넣어준다는 뜻
* 메모리는 별도 공간이 아니라 컨텍스트에 다시 들어가는 것 -> 길어지면 비싸짐
* 메모리 vs RAG
    * 둘 다 "모델에게 추가 정보를 넣어줌"은 동일
    * "정보의 출처"가 다른 것
    * RAG는 외부 문서에서 질문과 관련된 조각을 검색해 가져오고, 메모리는 이 대화의 과거를 복원해서 가져옴

### 6-1. 저장소와 열쇠
* checkpointer
    * 대화 상태를 저장하고 다시 불러오는 저장 장치
    * 여러 대화의 상태를 함께 저장할 수 있음

* thread_id
    * 그 저장소 안에서 특정 대화 흐름을 구분하는 키
    * 같은 값이면 같은 대화가 이어지고, 다른 값이면 다른 대화로 분리됨

* InMemorySaver - 실습용
    * LangGraph가 제공하는 체크포인터 구현
    * 메모리(RAM)에 저장해서 커널 재시작 및 파이썬 프로그램을 종료하면 전부 날아감
    * 실제 서비스에서는 ```SqliteSaver(파일에 저장)```나 ```PostgresSaver(DB에 저장)```을 사용 - 바꾸는 방법은 간단. checkpointer= 에 다른 객체를 넣으면 끝

In [ ]:
# checkpointer를 붙이면 같은 thread_id 안에서 대화 맥락을 이어갈 수 있음
from langgraph.checkpoint.memory import InMemorySaver # 메모리 안에 대화 상태 저장

memory_agent = create_agent(
    model=model, # 기억 유지할 에이전트
    tools=[get_korean_time],
    checkpointer=InMemorySaver(),
    system_prompt="당신은 학습 코치입니다. 사용자가 알려준 자기소개를 기억하세요.",
)

config = {"configurable": {"thread_id": "student-1"}} # 같은 thread_id면 같은 대화 흐름으로 이어짐
config.update(lf_config)

first = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 민수이고, 나는 개발자야."}]}, # 먼저 기억할 정보 저장
    config,
)
second = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "내 직업이 뭐였는지 기억해?"}]}, # 같은 thread_id로 후속 질문 보냄
    config,
)

print(second["messages"][-1].content)

## 7. 실습

### 7-1. A) 고객 문의를 티켓으로

문제: 고객 문의 문장을 읽고, 상담 시스템이 바로 저장할 수 있는 구조화 데이터로 바꾸는 에이전트를 만드세요.

* SupportTicket Pydantic 모델 정의. 필드는 customer_name, order_id, category, priority, summary
* category는 shipping, exchange, refund, other 중 하나
* priority는 low, medium, high 중 하나
* create_agent()에 response_format=SupportTicket 연결
* 결과는 result["structured_response"].model_dump()로 확인

==> 예시 입력: "안녕하세요. 박지민입니다. 주문 ORD-2040으로 받은 상품 색상이 잘못 왔습니다. 교환을 빨리 진행하고 싶어요."

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

class SupportTicket(BaseModel):
    customer_name: str = Field(discription="고객 이름"),
    order_id: str = Field(discription="주문 번호"),
    category: Literal["shipping", "exchange", "refund", "other"] = Field(discription="문의 유형"),
    priority: Literal["high", "medium", "low"] = Field(discription="처리 우선순위"),
    summary: str = Field(discription="문의 내용 요약")

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

ticket_agent = create_agent(
    model=model,
    tools=[],
    response_format=SupportTicket,
    system_prompt=(
        "사용자 문의에서 고객명, 주문번호, 문의 유형, 우선순위, 요약을 추출하세요. "
        "상품이 잘못 왔거나 교환 처리가 필요한 경우 category는 exchange로 분류하세요."
    )
)

result = ticket_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "안녕하세요. 박지민입니다. 주문 ORD-2040으로 받은 상품 색상이 잘못 왔습니다. 교환을 빨리 진행하고 싶어요."
            }
        ]
    }
)

ticket = result["structured_response"]
print(ticket.model_dump())
print(ticket.model_dump_json(indent=2))


/var/folders/qz/hyvszw9n4ms9wd1s5wn7fm2c0000gn/T/ipykernel_40740/3272551391.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  customer_name: str = Field(discription="고객 이름"),
/var/folders/qz/hyvszw9n4ms9wd1s5wn7fm2c0000gn/T/ipykernel_40740/3272551391.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  order_id: str = Field(discription="주문 번호"),
/var/folders/qz/hyvszw9n4ms9wd1s5wn7fm2c0000gn/T/ipykernel_40740/3272551391.py:9: PydanticDeprecatedSince20: Using extra keyword arguments on `Field`

{'customer_name': '박지민', 'order_id': 'ORD-2040', 'category': 'exchange', 'priority': 'high', 'summary': '상품 색상 교환 요청'}
{
  "customer_name": "박지민",
  "order_id": "ORD-2040",
  "category": "exchange",
  "priority": "high",
  "summary": "상품 색상 교환 요청"
}


### 7-2. B) 통합예제
문제: Tool + Structured Output + Memory를 함께 쓰는 에이전트를 완성하세요.

* 고객 문의 문장을 SupportRequest로 구조화한다
* 필요하면 주문 처리 가이드를 조회하는 Tool을 쓴다
* 같은 thread_id로 후속 요청까지 이어서 처리한다

In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class SupportRequest(BaseModel):
    order_id: str = Field(description="문의 대상 주문번호")
    request_summary: str = Field(description="사용자가 말한 문의 내용 요약")
    urgency: Literal["low", "medium", "high"] = Field(description="처리 긴급도")
    action_plan: str = Field(description="주문 처리 가이드와 사용잦 요청을 반영한 조치 요약")
    needs_human_review: bool = Field(description="고객지언팀이나 담당자 검토가 필요한지 여부")

@tool
def lookup_order_guide(order_id: str) -> str:
    """주문번호를 기준으로 처리 가이드가 필요할 때 사용합니다."""
    guides = {
        "ORD-1001": "ORD-1001은 배송 지연 상태입니다. 예상 도착일을 확인하고 지연 안내와 보상 쿠폰 여부를 검토합니다.",
        "ORD-1002": "ORD-1002는 사이즈 교환 문의입니다. 재고 확인 후 교환 가능 여부와 회수 일정을 안내합니다.",
        "ORD-1003": "ORD-1003은 환불 검수 대기 상태입니다. 검수 완료 후 결제 수단으로 환불됩니다.",
    }
    return guides.get(order_id, "해당 주문 가이드를 찾을 수 없습니다.")

integrated_agent = create_agent(
    model=model,
    tools=[lookup_order_guide],
    response_format=SupportRequest,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "사용자의 고객 문의를 SupportRequest 형식으로 정리하세요. "
        "주문별 처리 가이드가 필요하면 lookup_order_guide 도구를 사용하세요. "
        "긴급도 표현은 low, medium, high 중 하나로 변환하세요. "
        "사용자가 고객지원팀 검토를 요청하거나 긴급도가 높으면 needs_human_review를 true로 설정하세요. "
        "후속 요청에서 '그 주문'처럼 이전 대상을 가리키면 같은 thread_id의 이전 대화 내용을 이어서 사용하세요."
    ),
)

first_message = {
    "messages": [
        {
            "role": "user",
            "content": "주문 ORD-1001 배송이 아직 도착하지 않았어. 오늘 안으로 처리 요청을 만들고 싶어. 긴급도는 높음으로 보고, 필요하면 고객지원팀 검토로 넘겨줘.",
        }
    ]
}

followup_message = {
    "messages": [
        {
            "role": "user",
            "content": "그 주문은 오늘 오후 안에 고객에게 안내해야 해. 조치 요약에도 그 내용을 반영해줘.",
        }
    ]
}

practice_config = {"configurable": {"thread_id": "support-practice-1"}}

first_result = integrated_agent.invoke(first_message, practice_config)
print("[첫 번째 요청]")
print(first_result["structured_response"].model_dump())

followup_result = integrated_agent.invoke(followup_message, practice_config)
print("[후속 요청]")
print(followup_result["structured_response"].model_dump())

Deserializing unregistered type __main__.SupportRequest from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'SupportRequest')]


[첫 번째 요청]
{'order_id': 'ORD-1001', 'request_summary': '주문 ORD-1001 배송이 아직 도착하지 않았습니다. 오늘 안으로 처리 요청을 원합니다.', 'urgency': 'high', 'action_plan': '배송 지연 상태로, 예상 도착일을 확인하고 지연 안내 및 보상 쿠폰 여부를 검토합니다.', 'needs_human_review': True}
[후속 요청]
{'order_id': 'ORD-1001', 'request_summary': '주문 ORD-1001 배송이 아직 도착하지 않았습니다. 오늘 오후 안에 고객에게 안내해야 합니다.', 'urgency': 'high', 'action_plan': '배송 지연 상태로, 예상 도착일을 확인하고 지연 안내 및 보상 쿠폰 여부를 검토하여 오늘 오후 안에 고객에게 안내합니다.', 'needs_human_review': True}


# 치트시트 - 이것만 있으면 시작할 수 있음

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from typing import Literal

llm = ChanOpenAI(model="gpt-4o-mini", temperature=0)

# 1) Tool - 이름, 타입힌트, docstring이 모델에게 가는 설명서
@tool
def my_tool(order_id: str) -> str:
    """언제 이 도구를 써야 하는지 한 문장으로 작성"""
    return "결과 문자열" # 실패해도 None 말고 문장으로

# 2) 출력스키마 - 결과를 코드가 바로 쓸 수 있게
class MyResult(BaseModel):
    field_a: str = Field(description="이 칸의 의미")
    level: Literal["A", "B", "C"] = Field(description="등급")

# 3) 에이전트
agent = create_agent(
    model=llm,
    tools=[my_tool],
    response_format=MyResult,           # 선택
    checkpointer=InMemorySaver(),       # 선택
    system_prompt="역할과 Tool 사용 조건을 여기에 작성"
)

# 4) 실행
config = {"configurable": {"thread_id": "user-1"}} # checkpointer를 쓸 때만 필요
result = agent.invoke({"messages": [{"role": "user", "content": "질문"}]}, config)

# 5) 꺼내기
print(result["messages"][-1].content)               # 사람이 읽을 답
print(result["structured_response"].model_dump())   # 코드가 쓸 데이터